In [1]:
# Steps in Openseespy projects:
# 1. Units, Dimensions and Geometries.
# 2. Coordinates of nodes.
# 3. Support conditions.
# 4. Materials.
# 5. Sections.
# 6. Transfer geometries.
# 7. Elements.
# 8. Define floors (Deck or Slab).
# 9. Define Loads.
# 10. 

In [2]:
from openseespy import opensees as ops
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [3]:
# 1. Units
# Unints:
m = 1.0       # m
N = 1.0       # N
Pa = 1.0      # Pa
s = 1.0       # s

kg = N * s**2 / m

MPa = Pa * 1e6
mm = m * 1e-3
cm = m * 1e-2

g = 9.81     # m/s^2

In [4]:
# Initial commands for model
# -------------------------- Start Model -------------------------- #
# Clean All the variables...
ops.wipe()
# model('basic', '-ndm', ndm, '-ndf', ndf=ndm*(ndm+1)/2)
ops.model('basic', '-ndm', 3, '-ndf', 6)  # 3D with 6 DOFs.

In [5]:
# 2. Geometry and Node Coordinates
# -------------------------- Nodes -------------------------- #
# node(nodeTag, *crds, '-ndf', ndf, '-mass', *mass, '-disp', *disp, '-vel', *vel, '-accel', *accel)
# Generated Excel file from ETABS model ----> Geometry.xlsx
Node_Data = pd.read_excel("Geometry.xlsx", sheet_name="Node")
NONE_Nodes = pd.read_excel("Geometry.xlsx", sheet_name="NONE_POINTS")["UNIQUENAME"].to_list()

Node_Data = Node_Data[
    ~Node_Data["UniqueName"].isin(NONE_Nodes)
].reset_index(drop=True)

# Audits - Node
print("Number of nodes (filtered):", len(Node_Data))
print("Number of base nodes:", len(pd.read_excel("Geometry.xlsx", sheet_name="FIX_Node")))
print("Z levels:", sorted(Node_Data["Z (m)"].unique()))

for ID, X, Y, Z in zip(Node_Data["UniqueName"], Node_Data["X (m)"], Node_Data["Y (m)"], Node_Data["Z (m)"]):
    ops.node(ID, X, Y, Z)

Number of nodes (filtered): 1680
Number of base nodes: 46
Z levels: [np.float64(0.0), np.float64(6.204), np.float64(6.215), np.float64(7.1), np.float64(10.404), np.float64(10.415), np.float64(11.3), np.float64(14.604), np.float64(14.615), np.float64(15.5), np.float64(18.504), np.float64(18.515), np.float64(19.4), np.float64(22.404), np.float64(22.415), np.float64(23.3), np.float64(26.304), np.float64(26.315), np.float64(27.2), np.float64(30.904), np.float64(31.8)]


In [6]:
# 3. Support Conditions
# -------------------------- Fixed Support -------------------------- #
# fix(nodeTag, *constrValues)
Fix_Nodes = pd.read_excel("Geometry.xlsx", sheet_name="FIX_Node")

for ID in Fix_Nodes["UniqueName"]:
    ops.fix(ID, 1, 1, 1, 1, 1, 1)  # Fixed Support

# 3b. Diaphragm Masters (ETABS rigid diaphragm - Center of Mass)
# Story masses are diaphragm masses, not distributed independent masses
CenterMass_Data = pd.read_excel("Load.xlsx", sheet_name="Center of Mass")
Floor_Data = pd.read_excel("Geometry.xlsx", sheet_name="FLOOR_Connectivity")
# Masters at center of mass per diaphragm
Master_Tags = {}
next_tag = 9000
for story in ['Story1','Story2','Story3','Story4','Story5','Story6']:
    row = CenterMass_Data[CenterMass_Data["Story"]==story].iloc[0]
    tag = next_tag; next_tag += 1
    ops.node(tag, float(row["XCM (m)"]), float(row["YCM (m)"]), float(row["ZCM (m)"]))
    ops.fix(tag, 0, 0, 1, 1, 1, 1)  # UX,UY free (lateral), UZ,RX,RY,RZ fixed
    Master_Tags[story] = tag
for d in ['D7-1','D7-2']:
    row = CenterMass_Data[CenterMass_Data["Diaphragm"]==d].iloc[0]
    tag = next_tag; next_tag += 1
    ops.node(tag, float(row["XCM (m)"]), float(row["YCM (m)"]), float(row["ZCM (m)"]))
    ops.fix(tag, 0, 0, 1, 1, 1, 1)
    Master_Tags[d] = tag
print("Diaphragm masters:", Master_Tags)

# 3c. Constraints for other nodes - lateral model with secondary fixing
# Determine primary floor nodes (deg>1) vs secondary/hanging
from collections import Counter
Deg_Count = Counter()
Col_Ele_Data = pd.read_excel("Geometry.xlsx", sheet_name="COLUMN_Connectivity")
Beam_Ele_Data = pd.read_excel("Geometry.xlsx", sheet_name="BEAM_Connectivity")
Brace_Ele_Data = pd.read_excel("Geometry.xlsx", sheet_name="BRACE_Connectivity")
for _, r in Col_Ele_Data.iterrows():
    Deg_Count[int(r["UniquePtI"])] += 1
    Deg_Count[int(r["UniquePtJ"])] += 1
for _, r in Beam_Ele_Data.iterrows():
    Deg_Count[int(r["UniquePtI"])] += 1
    Deg_Count[int(r["UniquePtJ"])] += 1
for _, r in Brace_Ele_Data.iterrows():
    Deg_Count[int(r["UniquePtI"])] += 1
    Deg_Count[int(r["UniquePtJ"])] += 1
Floor_Nodes = set(Floor_Data["UniquePt1"].tolist() + Floor_Data["UniquePt2"].tolist() + Floor_Data["UniquePt3"].tolist() + Floor_Data["UniquePt4"].tolist())
Floor_Nodes = Floor_Nodes & set(Node_Data["UniqueName"])
Other_Nodes = Node_Data["UniqueName"].to_list()
for i in Fix_Nodes["UniqueName"]:
    if i in Other_Nodes:
        Other_Nodes.remove(i)
primary_cnt = 0
secondary_cnt = 0
for nid in Other_Nodes:
    nid = int(nid)
    if nid in Floor_Nodes and Deg_Count[nid] > 1:
        ops.fix(nid, 0, 0, 1, 1, 1, 1)  # Primary floor - UX,UY free
        primary_cnt += 1
    else:
        ops.fix(nid, 1, 1, 1, 1, 1, 1)  # Secondary/hanging - fully fixed
        secondary_cnt += 1
print("Primary floor free (UX,UY):", primary_cnt)
print("Secondary fully fixed:", secondary_cnt)
# Diaphragm constraints - equalDOF UX,UY
for story in ['Story1','Story2','Story3','Story4','Story5','Story6']:
    slaves = set(Floor_Data[Floor_Data["Story"]==story]["UniquePt1"].tolist() + Floor_Data[Floor_Data["Story"]==story]["UniquePt2"].tolist() + Floor_Data[Floor_Data["Story"]==story]["UniquePt3"].tolist() + Floor_Data[Floor_Data["Story"]==story]["UniquePt4"].tolist())
    slaves = [int(n) for n in slaves if n in Floor_Nodes and Deg_Count[n] > 1]
    for s in slaves:
        ops.equalDOF(Master_Tags[story], s, 1, 2)
    print(f"Diaphragm {story} {len(slaves)} slaves -> master {Master_Tags[story]}")
khar = Floor_Data[Floor_Data["Story"]=="Kharposhteh"]
all_khar = set(khar["UniquePt1"].tolist() + khar["UniquePt2"].tolist() + khar["UniquePt3"].tolist() + khar["UniquePt4"].tolist())
all_khar = [n for n in all_khar if n in Floor_Nodes and Deg_Count[n] > 1]
# Split Kharposhteh by X<25 (west) vs east (ETABS D7-1/D7-2)
west = [n for n in all_khar if float(Node_Data[Node_Data["UniqueName"]==n]["X (m)"].iloc[0]) < 25]
east = [n for n in all_khar if float(Node_Data[Node_Data["UniqueName"]==n]["X (m)"].iloc[0]) >= 25]
for s in west:
    ops.equalDOF(Master_Tags['D7-1'], s, 1, 2)
for s in east:
    ops.equalDOF(Master_Tags['D7-2'], s, 1, 2)
print(f"Kharposhteh W {len(west)} -> D7-1, E {len(east)} -> D7-2")

Diaphragm masters: {'Story1': 9000, 'Story2': 9001, 'Story3': 9002, 'Story4': 9003, 'Story5': 9004, 'Story6': 9005, 'D7-1': 9006, 'D7-2': 9007}
Primary floor free (UX,UY): 393
Secondary fully fixed: 1241
Diaphragm Story1 52 slaves -> master 9000
Diaphragm Story2 49 slaves -> master 9001
Diaphragm Story3 44 slaves -> master 9002
Diaphragm Story4 73 slaves -> master 9003
Diaphragm Story5 78 slaves -> master 9004
Diaphragm Story6 72 slaves -> master 9005
Kharposhteh W 13 -> D7-1, E 12 -> D7-2


In [7]:
# Model 2 - ElasticBeamColumn
# 4. Material

# Steel Material Properties:
# uniaxialMaterial('Steel02', matTag, Fy, E0, b, *params, a1=a2*Fy/E0, a2=1.0, a3=a4*Fy/E0, a4=1.0, sigInit=0.0)
fy = 240 * MPa          # Steel ----> Based on ETABS model
fye = 1.1 * fy
fu = 370 * MPa
fue = 1.1 * fu
E_steel = 210e9
# E_steel = 1e20  # Original bug - too stiff
nu_steel = 0.3
G_steel = E_steel /(2 * (1+ nu_steel))
rho_steel = 7850 * kg/m**3
b = 0.01  # Strain hardening ratio for Steel02 material
params = [15.0, 0.925, 0.15]  # Parameters for Steel02 material
a2 = 1.0
a1 = a2 * fy / E_steel
a4 = 1.0
a3 = a4 * fy / E_steel
sigInit = 0.0
MAT_TAG_STEEL = 1
ops.uniaxialMaterial('Steel02', MAT_TAG_STEEL, fy, E_steel, b, *params, a1, a2, a3, a4, sigInit)

In [8]:
# 5. Section
# Section Properties: (based on ETABS model)
Section_Data = pd.read_excel("Geometry.xlsx", sheet_name="SECTION_Properties")

In [9]:
# 5. Sections ---> Real Sections for Nonlinear Analysis:
# beamIntegration('Lobatto', tag, secTag, N)


# section('WFSection2d', secTag, matTag, d, tw, bf, tf, Nfw, Nff)
# I - Sections:

def I_Section(d, tw, bf, tf, J,secTag, G = G_steel):
    ops.section('Fiber', secTag, '-GJ', G * J)

    X_bar = (2 * bf * tf * bf/2 + tw * (d - 2 * tf) * tw/2)/(2 * bf * tf + tw * (d - 2 * tf))
    Y_bar = d/2

    # Bottom Flange
    crdsI_1 = [-X_bar * 1e-3, -Y_bar * 1e-3]
    crdsJ_1 = [X_bar * 1e-3, (-Y_bar + tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL,6, 2, *crdsI_1, *crdsJ_1)

    # Web
    crdsI_2 = [-tw/2 * 1e-3, (-Y_bar + tf) * 1e-3]
    crdsJ_2 = [tw/2 * 1e-3, (Y_bar - tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL,2, 6, *crdsI_2, *crdsJ_2)

    # Top Flange
    crdsI_3 = [-X_bar * 1e-3, (Y_bar - tf) * 1e-3]
    crdsJ_3 = [X_bar * 1e-3, Y_bar * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL,6, 2, *crdsI_3, *crdsJ_3)

    ops.beamIntegration('Lobatto', secTag, secTag, 5)


# IPN 120
J = Section_Data.loc[Section_Data["Name"] == "IPN120", 'J (m^4)'].values[0]
I_Section(120, 5.1, 58, 7.7, J,1, G_steel)

# IPN 140
J = Section_Data.loc[Section_Data["Name"] == "IPN140", 'J (m^4)'].values[0]
I_Section(140, 5.7, 66, 8.6, J,2, G_steel)

# IPN 160
J = Section_Data.loc[Section_Data["Name"] == "IPN160", 'J (m^4)'].values[0]
I_Section(160, 6.3, 74, 9.5, J,3, G_steel)

# IPN 180
J = Section_Data.loc[Section_Data["Name"] == "IPN180", 'J (m^4)'].values[0]
I_Section(180, 6.9, 82, 10.4, J,4, G_steel)

# IPN 200
J = Section_Data.loc[Section_Data["Name"] == "IPN200", 'J (m^4)'].values[0]
I_Section(200, 7.5, 90, 11.3, J,5, G_steel)

# IPN 220
J = Section_Data.loc[Section_Data["Name"] == "IPN220", 'J (m^4)'].values[0]
I_Section(220, 8.1, 98, 12.2, J,6, G_steel)

# IPN 240
J = Section_Data.loc[Section_Data["Name"] == "IPN240", 'J (m^4)'].values[0]
I_Section(240, 8.7, 106, 13.1, J,7, G_steel)

# IPN 260
J = Section_Data.loc[Section_Data["Name"] == "IPN260", 'J (m^4)'].values[0]
I_Section(260, 9.4, 113, 14.1, J,8, G_steel)

# IPN 280
J = Section_Data.loc[Section_Data["Name"] == "IPN280", 'J (m^4)'].values[0]
I_Section(280, 10.1, 119, 15.2, J,9, G_steel)



# Box - Sections:
# section('Fiber', secTag, '-GJ', GJ)
# patch('rect', matTag, numSubdivY, numSubdivZ, *crdsI, *crdsJ)

def Box_Section(Width, thickness, secTag, J, G = G_steel):
    ops.section('Fiber', secTag, '-GJ', G * J)

    # Left Rectangle
    crdsI_1 = [-Width/2 * 1e-3, -Width/2 * 1e-3]
    crdsJ_1 = [(-Width/2 + thickness) * 1e-3, Width/2 * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_1, *crdsJ_1)

    # Bottom Rectangle
    crdsI_2 = [(-Width/2 + thickness) * 1e-3, -Width/2 * 1e-3]
    crdsJ_2 = [(Width/2 - thickness) * 1e-3, (-Width/2 + thickness) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_2, *crdsJ_2)

    # Top Rectangle
    crdsI_3 = [(-Width/2 + thickness) * 1e-3, (Width/2 - thickness) * 1e-3]
    crdsJ_3 = [(Width/2 - thickness) * 1e-3, Width/2 * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_3, *crdsJ_3)

    # Right Rectangle
    crdsI_4 = [(Width/2 - thickness) * 1e-3, -Width/2 * 1e-3]
    crdsJ_4 = [Width/2 * 1e-3, Width/2 * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_4, *crdsJ_4)

    ops.beamIntegration('Lobatto', secTag, secTag, 5)

# A-(BOX350x18)
J = Section_Data.loc[Section_Data["Name"] == "A-(BOX350x18)", 'J (m^4)'].values[0]
Box_Section(350, 18, 11, J, G = G_steel)

# B-(BOX300x18)
J = Section_Data.loc[Section_Data["Name"] == "B-(BOX300x18)", 'J (m^4)'].values[0]
Box_Section(300, 18, 12, J, G = G_steel)

# C-(BOX250x18)
J = Section_Data.loc[Section_Data["Name"] == "C-(BOX250x18)", 'J (m^4)'].values[0]
Box_Section(250, 18, 13, J, G = G_steel)

# D-(BOX200x18)
J = Section_Data.loc[Section_Data["Name"] == "D-(BOX200x18)", 'J (m^4)'].values[0]
Box_Section(200, 18, 14, J, G = G_steel)

# E-(BOX200x12)
J = Section_Data.loc[Section_Data["Name"] == "E-(BOX200x12)", 'J (m^4)'].values[0]
Box_Section(200, 12, 15, J, G = G_steel)


# L - Sections:
# section('Fiber', secTag, '-GJ', GJ)
# patch('rect', matTag, numSubdivY, numSubdivZ, *crdsI, *crdsJ)

def L_Section(Width, thickness, secTag, J, G = G_steel):
    X_bar = (Width * thickness * Width/2 + (Width - thickness) * thickness * thickness/2)/(Width * thickness + (Width - thickness) * thickness)
    Y_bar = (Width * thickness * thickness/2 + (Width - thickness) * thickness * ((Width - thickness)/2 + thickness))/(Width * thickness + (Width - thickness) * thickness)

    ops.section('Fiber', secTag, '-GJ', G * J)

    # Left Rectangle
    crdsI_1 = [-X_bar * 1e-3, -Y_bar * 1e-3]
    crdsJ_1 = [(-X_bar + thickness) * 1e-3, (Width - Y_bar) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_1, *crdsJ_1)

    # Bottom Rectangle
    crdsI_2 = [(-X_bar + thickness) * 1e-3, -Y_bar * 1e-3]
    crdsJ_2 = [(Width - X_bar) * 1e-3, (-X_bar + thickness) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_2, *crdsJ_2)

    ops.beamIntegration('Lobatto', secTag, secTag, 5)

# L80X8
J = Section_Data.loc[Section_Data["Name"] == "L80X8", 'J (m^4)'].values[0]
L_Section(80, 8, 101, J, G = G_steel)


# 2I - Sections:
# section('Fiber', secTag, '-GJ', GJ)
# patch('rect', matTag, numSubdivY, numSubdivZ, *crdsI, *crdsJ)

def D_I(bf, tf, hw, tw, secTag, J, G = G_steel):
    X_bar = bf
    d = hw + 2 * tf
    Y_bar = d/2

    ops.section('Fiber', secTag, '-GJ', G * J)

    # Left Bottom Flange
    crdsI_1 = [-X_bar * 1e-3, -Y_bar * 1e-3]
    crdsJ_1 = [0.0 * 1e-3, (-Y_bar + tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_1, *crdsJ_1)

    # Left Web
    crdsI_2 = [(-X_bar + (bf - tw)/2) * 1e-3, (-Y_bar + tf) * 1e-3]
    crdsJ_2 = [(-X_bar + (bf - tw)/2 + tw) * 1e-3, (Y_bar - tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_2, *crdsJ_2)

    # Left Top Flange
    crdsI_3 = [-X_bar * 1e-3, (Y_bar - tf) * 1e-3]
    crdsJ_3 = [0.0 * 1e-3, Y_bar * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_3, *crdsJ_3)

    # Right Bottom Flange
    crdsI_4 = [0.0 * 1e-3, -Y_bar * 1e-3]
    crdsJ_4 = [X_bar * 1e-3, (-Y_bar + tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_4, *crdsJ_4)

    # Right Web
    crdsI_5 = [(X_bar - (bf - tw)/2 - tw) * 1e-3, (-Y_bar + tf) * 1e-3]
    crdsJ_5 = [(X_bar - (bf - tw)/2) * 1e-3, (Y_bar - tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_5, *crdsJ_5)

    # Right Top Flange
    crdsI_6 = [0.0 * 1e-3, (Y_bar - tf) * 1e-3]
    crdsJ_6 = [X_bar * 1e-3, Y_bar * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_6, *crdsJ_6)

    ops.beamIntegration('Lobatto', secTag, secTag, 5)

# BI_(2PL400x18+2FL200x18)
J = Section_Data.loc[Section_Data["Name"] == "BI_(2PL400x18+2FL200x18)", 'J (m^4)'].values[0]
D_I(200, 18, 400, 18, 1001, J, G = G_steel)

# BII_(2PL300x18+2FL200x18)
J = Section_Data.loc[Section_Data["Name"] == "BII_(2PL300x18+2FL200x18)", 'J (m^4)'].values[0]
D_I(200, 18, 300, 18, 1002, J, G = G_steel)

# 2INP180
J = Section_Data.loc[Section_Data["Name"] == "2INP180", 'J (m^4)'].values[0]
D_I(82, 10.4, 159.2, 6.9, 1003, J, G = G_steel)

# 2INP220
J = Section_Data.loc[Section_Data["Name"] == "2INP220", 'J (m^4)'].values[0]
D_I(98, 12.2, 195.6, 8.1, 1004, J, G = G_steel)

# 2INP240
J = Section_Data.loc[Section_Data["Name"] == "2INP240", 'J (m^4)'].values[0]
D_I(106, 13.1, 213.8, 8.7, 1005, J, G = G_steel)

# 2INP260
J = Section_Data.loc[Section_Data["Name"] == "2INP260", 'J (m^4)'].values[0]
D_I(113, 14.1, 231.8, 9.4, 1006, J, G = G_steel)

# 2IPE200
J = Section_Data.loc[Section_Data["Name"] == "2IPE200", 'J (m^4)'].values[0]
D_I(100, 8.5, 183, 5.6, 1007, J, G = G_steel)


# 2L - Sections:
# section('Fiber', secTag, '-GJ', GJ)
# patch('rect', matTag, numSubdivY, numSubdivZ, *crdsI, *crdsJ)

def D_L(Width, thickness, space, secTag, J, G = G_steel):
    X_bar = (2 * Width + space)/2
    Y_bar = (Width * thickness * thickness/2 + (Width - thickness) * thickness * ((Width - thickness)/2 + thickness))/(Width * thickness + (Width - thickness) * thickness)

    ops.section('Fiber', secTag, '-GJ', G * J)

    # Left Horizental Rectangle
    crdsI_1 = [-X_bar * 1e-3, -Y_bar * 1e-3]
    crdsJ_1 = [-space/2 * 1e-3, (-Y_bar + thickness) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_1, *crdsJ_1)

    # Left Vertical Rectangle
    crdsI_2 = [(-space/2 - thickness) * 1e-3, (-Y_bar + thickness) * 1e-3]
    crdsJ_2 = [-space/2 * 1e-3, (Width - Y_bar) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_2, *crdsJ_2)

    # Right Horizental Rectangle
    crdsI_3 = [space/2 * 1e-3, -Y_bar * 1e-3]
    crdsJ_3 = [X_bar * 1e-3, (-Y_bar + thickness) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_3, *crdsJ_3)

    # Right Vertical Rectangle
    crdsI_4 = [space/2 * 1e-3, (-Y_bar + thickness) * 1e-3]
    crdsJ_4 = [(space/2 + thickness) * 1e-3, (Width - Y_bar) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_4, *crdsJ_4)

    ops.beamIntegration('Lobatto', secTag, secTag, 5)

# 2L120X12/10/
J = Section_Data.loc[Section_Data["Name"] == "2L120X12/10/", 'J (m^4)'].values[0]
D_L(120, 12, 10, 10001, J, G = G_steel)

# 2L50X5/10/
J = Section_Data.loc[Section_Data["Name"] == "2L50X5/10/", 'J (m^4)'].values[0]
D_L(50, 5, 10, 10002, J, G = G_steel)

# 2L60X6/10/
J = Section_Data.loc[Section_Data["Name"] == "2L60X6/10/", 'J (m^4)'].values[0]
D_L(60, 6, 10, 10003, J, G = G_steel)

# 2L70X7/10/
J = Section_Data.loc[Section_Data["Name"] == "2L70X7/10/", 'J (m^4)'].values[0]
D_L(70, 7, 10, 10004, J, G = G_steel)

# 2L80X8/10/
J = Section_Data.loc[Section_Data["Name"] == "2L80X8/10/", 'J (m^4)'].values[0]
D_L(80, 8, 10, 10005, J, G = G_steel)

# 2L90X9/10/
J = Section_Data.loc[Section_Data["Name"] == "2L90X9/10/", 'J (m^4)'].values[0]
D_L(90, 9, 10, 10006, J, G = G_steel)

# 2UPN Sections:

def D_UPN(bf, tf, d, tw, s, secTag, J,G = G_steel):
    X_bar = (2 * bf + s)/2
    Y_bar = d/2
    ops.section('Fiber', secTag, '-GJ', G * J)
    # Left Bottom Flange
    crdsI_1 = [-X_bar * 1e-3, -Y_bar * 1e-3]
    crdsJ_1 = [-s/2 * 1e-3, (-Y_bar + tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_1, *crdsJ_1)
    # Left Web
    crdsI_2 = [(-s/2 - tw) * 1e-3, (-Y_bar + tf) * 1e-3]
    crdsJ_2 = [-s/2 * 1e-3, (Y_bar - tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_2, *crdsJ_2)
    # Left Top Flange
    crdsI_3 = [-X_bar * 1e-3, (Y_bar - tf) * 1e-3]
    crdsJ_3 = [-s/2 * 1e-3, Y_bar * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_3, *crdsJ_3)
    # Right Bottom Flange
    crdsI_4 = [s/2 * 1e-3, -Y_bar * 1e-3]
    crdsJ_4 = [X_bar * 1e-3, (-Y_bar + tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_4, *crdsJ_4)
    # Right Web
    crdsI_5 = [s/2 * 1e-3, (-Y_bar + tf) * 1e-3]
    crdsJ_5 = [(s/2 + tw) * 1e-3, (Y_bar - tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_5, *crdsJ_5)
    # Right Top Flange
    crdsI_6 = [s/2 * 1e-3, (Y_bar - tf) * 1e-3]
    crdsJ_6 = [X_bar * 1e-3, Y_bar * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_6, *crdsJ_6)

    ops.beamIntegration('Lobatto', secTag, secTag, 5)


# 2UPN100/20/
J = Section_Data.loc[Section_Data["Name"] == "2UPN100/20/", 'J (m^4)'].values[0]
bf = 50
tf = 8.5
d = 100
tw = 6
s = 20
secTag = 100001
D_UPN(bf, tf, d, tw, s, secTag, J, G = G_steel)

# 2UPN120/20/
J = Section_Data.loc[Section_Data["Name"] == "2UPN120/20/", 'J (m^4)'].values[0]
bf = 55
tf = 9
d = 120
tw = 7
s = 20
secTag = 100002
D_UPN(bf, tf, d, tw, s, secTag, J, G = G_steel)

# 2UPN140/10/
J = Section_Data.loc[Section_Data["Name"] == "2UPN140/10/", 'J (m^4)'].values[0]
bf = 60
tf = 10
d = 140
tw = 7
s = 10
secTag = 100003
D_UPN(bf, tf, d, tw, s, secTag, J, G = G_steel)

# 2UPN140/20/
J = Section_Data.loc[Section_Data["Name"] == "2UPN140/20/", 'J (m^4)'].values[0]
bf = 60
tf = 10
d = 140
tw = 7
s = 20
secTag = 100004
D_UPN(bf, tf, d, tw, s, secTag, J, G = G_steel)

# 2UPN160/20/
J = Section_Data.loc[Section_Data["Name"] == "2UPN160/20/", 'J (m^4)'].values[0]
bf = 65
tf = 10.5
d = 160
tw = 7.5
s = 20
secTag = 100005
D_UPN(bf, tf, d, tw, s, secTag, J, G = G_steel)

# 2UPN180/20/
J = Section_Data.loc[Section_Data["Name"] == "2UPN180/20/", 'J (m^4)'].values[0]
bf = 70
tf = 11
d = 180
tw = 8
s = 20
secTag = 100006
D_UPN(bf, tf, d, tw, s, secTag, J, G = G_steel)

# 2UPN180/10/
J = Section_Data.loc[Section_Data["Name"] == "2UPN180/10/", 'J (m^4)'].values[0]
bf = 70
tf = 11
d = 180
tw = 8
s = 10
secTag = 100007
D_UPN(bf, tf, d, tw, s, secTag, J, G = G_steel)

# 2UPN200/20/
J = Section_Data.loc[Section_Data["Name"] == "2UPN200/20/", 'J (m^4)'].values[0]
bf = 75
tf = 11.5
d = 200
tw = 8.5
s = 20
secTag = 100008
D_UPN(bf, tf, d, tw, s, secTag, J, G = G_steel)

# 2UPN220/20/
J = Section_Data.loc[Section_Data["Name"] == "2UPN220/20/", 'J (m^4)'].values[0]
bf = 80
tf = 12.5
d = 220
tw = 9
s = 20
secTag = 100009
D_UPN(bf, tf, d, tw, s, secTag, J, G = G_steel)

# 2UPN Side by Side Sections:

def D_UPN_Side_by_Side(bf, tf, d, tw, secTag, J,G = G_steel):
    X_bar = bf
    Y_bar = d/2
    ops.section('Fiber', secTag, '-GJ', G * J)
    # Left Bottom Flange
    crdsI_1 = [-X_bar * 1e-3, -Y_bar * 1e-3]
    crdsJ_1 = [0.0 * 1e-3, (-Y_bar + tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_1, *crdsJ_1)
    # Left Web
    crdsI_2 = [-X_bar * 1e-3, (-Y_bar + tf) * 1e-3]
    crdsJ_2 = [(-X_bar + tw) * 1e-3, (Y_bar - tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_2, *crdsJ_2)
    # Left Top Flange
    crdsI_3 = [-X_bar * 1e-3, (Y_bar - tf) * 1e-3]
    crdsJ_3 = [0.0 * 1e-3, Y_bar * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_3, *crdsJ_3)
    # Right Bottom Flange
    crdsI_4 = [0.0 * 1e-3, -Y_bar * 1e-3]
    crdsJ_4 = [X_bar * 1e-3, (-Y_bar + tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_4, *crdsJ_4)
    # Right Web
    crdsI_5 = [(X_bar - tw) * 1e-3, (-Y_bar + tf) * 1e-3]
    crdsJ_5 = [X_bar * 1e-3, (Y_bar - tf) * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 2, 6, *crdsI_5, *crdsJ_5)
    # Right Top Flange
    crdsI_6 = [0.0 * 1e-3, (Y_bar - tf) * 1e-3]
    crdsJ_6 = [X_bar * 1e-3, Y_bar * 1e-3]
    ops.patch('rect', MAT_TAG_STEEL, 6, 2, *crdsI_6, *crdsJ_6)

    ops.beamIntegration('Lobatto', secTag, secTag, 5)


# 2UNP120/SIDEBYSIDE
J = Section_Data.loc[Section_Data["Name"] == "2UNP120/SIDEBYSIDE", 'J (m^4)'].values[0]
bf = 55
tf = 9
d = 120
tw = 7
secTag = 1000001
D_UPN_Side_by_Side(bf, tf, d, tw, secTag, J, G = G_steel)

In [10]:
# 6. Geometry Transform
def get_vecxz(node_i, node_j):
    """
    Calculate a suitable vecxz vector for a 3D OpenSees element.


    node_i, node_j:
        Global coordinates [X, Y, Z]

    Returns:
        vecxz = [vx, vy, vz]
    """

    xi = np.array(node_i, dtype=float)
    xj = np.array(node_j, dtype=float)

    # Local x-axis of the element
    x_local = xj - xi
    L = np.linalg.norm(x_local)

    if L == 0:
        raise ValueError("Element has zero length.")

    x_local = x_local / L

    # Candidate global reference vectors
    global_axes = [
        np.array([1.0, 0.0, 0.0]),  # Global X
        np.array([0.0, 1.0, 0.0]),  # Global Y
        np.array([0.0, 0.0, 1.0])   # Global Z
    ]

    # Choose the global axis that is LEAST parallel to local x
    dots = [abs(np.dot(x_local, axis)) for axis in global_axes]

    vecxz = global_axes[np.argmin(dots)]

    return vecxz.tolist()

In [11]:
# 7. Define Elements
# Col_Ele_Data, Beam_Ele_Data, Brace_Ele_Data already read for constraints
Section_Assign_Data = pd.read_excel("Geometry.xlsx", sheet_name="FRAME_Sections")
Section_Properties_Data = pd.read_excel("Geometry.xlsx", sheet_name="SECTION_Properties")
Frame_Release_Data = pd.read_excel("Geometry.xlsx", sheet_name="FRAME_Releases&PartialFixity")["UniqueName"]
# Correct section mapping: UniqueName -> Analysis Section (was Label -> first)
Sec_Map = dict(zip(Section_Assign_Data["UniqueName"], Section_Assign_Data["Analysis Section"]))
# geomTransf(transfType, transfTag, *transfArgs)
# element('forceBeamColumn', eleTag, *eleNodes, transfTag, integrationTag, '-iter', maxIter=10, tol=1e-12, '-mass', mass=0.0)
# element('Truss', eleTag, *eleNodes, A, matTag, <'-rho', rho>, <'-cMass', cFlag>, <'-doRayleigh', rFlag>)

transfType_col = 'PDelta'
transfType_beam = 'Linear'
transfType_brace_and_truss = 'PDelta'

eleTag_Column = 0
eleTag_Beam = 1000
eleTag_Brace_and_Truss = 5000



for i in range(Col_Ele_Data.shape[0]): 

    # Create Section Properties for Column:
    section_name = Sec_Map[int(Col_Ele_Data.loc[i, "UniqueName"])]
    eleNodes = [int(Col_Ele_Data.loc[i, "UniquePtI"]), int(Col_Ele_Data.loc[i, "UniquePtJ"])]
    
    SecTag = int(Section_Properties_Data.loc[Section_Properties_Data["Name"] == section_name, "SecTag"].iloc[0])
    

    # Columns: always forceBeamColumn (release is pinned-pinned, not Truss)
    eleTag_Column += 1
    UniquePtI = int(Col_Ele_Data.loc[i, "UniquePtI"])
    UniquePtJ = int(Col_Ele_Data.loc[i, "UniquePtJ"])
    node_i_coords = Node_Data.loc[Node_Data["UniqueName"] == UniquePtI, ["X (m)", "Y (m)", "Z (m)"]].values.flatten()
    node_j_coords = Node_Data.loc[Node_Data["UniqueName"] == UniquePtJ, ["X (m)", "Y (m)", "Z (m)"]].values.flatten()
    nodes = {
        UniquePtI: node_i_coords,
        UniquePtJ: node_j_coords
    }
    node_i = nodes[UniquePtI]
    node_j = nodes[UniquePtJ]
    vecxz_col = get_vecxz(node_i, node_j)
    transfTag_col = eleTag_Column
    ops.geomTransf(transfType_col, transfTag_col, *vecxz_col)
    ops.element('forceBeamColumn', eleTag_Column, *eleNodes, transfTag_col, SecTag,'-iter', 10, 1e-5)

for j in range(Beam_Ele_Data.shape[0]):
    
    # Create Section Properties for Beam:
    section_name = Sec_Map[int(Beam_Ele_Data.loc[j, "UniqueName"])]
    eleNodes = [int(Beam_Ele_Data.loc[j, "UniquePtI"]), int(Beam_Ele_Data.loc[j, "UniquePtJ"])]

    SecTag = int(Section_Properties_Data.loc[Section_Properties_Data["Name"] == section_name, "SecTag"].iloc[0])

    # Beams: always forceBeamColumn
    eleTag_Beam += 1
    UniquePtI = int(Beam_Ele_Data.loc[j, "UniquePtI"])
    UniquePtJ = int(Beam_Ele_Data.loc[j, "UniquePtJ"])
    node_i_coords = Node_Data.loc[Node_Data["UniqueName"] == UniquePtI, ["X (m)", "Y (m)", "Z (m)"]].values.flatten()
    node_j_coords = Node_Data.loc[Node_Data["UniqueName"] == UniquePtJ, ["X (m)", "Y (m)", "Z (m)"]].values.flatten()
    nodes = {
        UniquePtI: node_i_coords,
        UniquePtJ: node_j_coords
    }
    node_i = nodes[UniquePtI]
    node_j = nodes[UniquePtJ]
    vecxz_beam = get_vecxz(node_i, node_j)
    transfTag_beam =  eleTag_Beam
    ops.geomTransf(transfType_beam, transfTag_beam, *vecxz_beam)
    ops.element('forceBeamColumn', eleTag_Beam, *eleNodes, transfTag_beam, SecTag,'-iter', 10, 1e-5)

for k in range(Brace_Ele_Data.shape[0]):
    eleTag_Brace_and_Truss += 1
    eleName = (Brace_Ele_Data.loc[k, "BraceBay"])
    section_name = Sec_Map[int(Brace_Ele_Data.loc[k, "UniqueName"])]
    eleNodes = [int(Brace_Ele_Data.loc[k, "UniquePtI"]), int(Brace_Ele_Data.loc[k, "UniquePtJ"])]
    brace_area = Section_Properties_Data.loc[Section_Properties_Data["Name"] == section_name, "Area (m^2)"].iloc[0]

    # Braces: always Truss (two-force, M+T released)
    cFlag = 0
    rFlag = 0
    ops.element('Truss', eleTag_Brace_and_Truss, *eleNodes, brace_area, MAT_TAG_STEEL, '-rho', rho_steel, '-cMass', cFlag, '-doRayleigh', rFlag)

print("Number of elements:", len(ops.getEleTags()))
print("Columns:", len(Col_Ele_Data), "Beams:", len(Beam_Ele_Data), "Braces:", len(Brace_Ele_Data))

Number of elements: 2157
Columns: 667 Beams: 813 Braces: 677


In [12]:
# ============================================================
# 3D INTERACTIVE OPENSEES MODEL VIEWER
# ============================================================

import plotly.graph_objects as go


def plot_opensees_3d(
    show_nodes=True,
    show_node_labels=False,
    show_element_labels=False,
    node_size=3,
    line_width=4,
    show_supports=True
):

    # --------------------------------------------------------
    # 1. GET NODES
    # --------------------------------------------------------

    node_tags = ops.getNodeTags()

    node_coords = {}
    
    for node in node_tags:
        try:
            crd = ops.nodeCoord(node)

            # فقط 3D nodes
            if len(crd) >= 3:
                node_coords[node] = np.array(crd[:3], dtype=float)

        except:
            pass

    print(f"Number of nodes: {len(node_coords)}")


    # --------------------------------------------------------
    # 2. GET ELEMENTS
    # --------------------------------------------------------

    ele_tags = ops.getEleTags()

    print(f"Number of elements: {len(ele_tags)}")


    # --------------------------------------------------------
    # 3. CREATE FIGURE
    # --------------------------------------------------------

    fig = go.Figure()


    # --------------------------------------------------------
    # 4. ELEMENTS
    # --------------------------------------------------------

    for ele in ele_tags:

        try:
            nodes = ops.eleNodes(ele)

            if len(nodes) < 2:
                continue

            n1 = nodes[0]
            n2 = nodes[1]

            if n1 not in node_coords or n2 not in node_coords:
                continue

            x1, y1, z1 = node_coords[n1]
            x2, y2, z2 = node_coords[n2]

            # --------------------------------------------
            # ELEMENT LINE
            # --------------------------------------------

            fig.add_trace(
                go.Scatter3d(
                    x=[x1, x2],
                    y=[y1, y2],
                    z=[z1, z2],

                    mode="lines",

                    line=dict(
                        width=line_width
                    ),

                    hovertemplate=(
                        f"<b>Element {ele}</b><br>"
                        f"Node I: {n1}<br>"
                        f"Node J: {n2}<br>"
                        f"Length: "
                        f"{np.linalg.norm(node_coords[n2]-node_coords[n1]):.3f} m"
                        "<extra></extra>"
                    ),

                    showlegend=False
                )
            )


            # --------------------------------------------
            # ELEMENT LABEL
            # --------------------------------------------

            if show_element_labels:

                xm = (x1 + x2) / 2
                ym = (y1 + y2) / 2
                zm = (z1 + z2) / 2

                fig.add_trace(
                    go.Scatter3d(
                        x=[xm],
                        y=[ym],
                        z=[zm],

                        mode="text",

                        text=[str(ele)],

                        textfont=dict(
                            size=9
                        ),

                        showlegend=False
                    )
                )

        except Exception:
            continue


    # --------------------------------------------------------
    # 5. NODES
    # --------------------------------------------------------

    if show_nodes:

        X = []
        Y = []
        Z = []
        labels = []

        for node, coord in node_coords.items():

            X.append(coord[0])
            Y.append(coord[1])
            Z.append(coord[2])

            labels.append(str(node))


        fig.add_trace(
            go.Scatter3d(

                x=X,
                y=Y,
                z=Z,

                mode="markers",

                marker=dict(
                    size=node_size
                ),

                text=labels,

                hovertemplate=(
                    "<b>Node %{text}</b><br>"
                    "X = %{x:.3f} m<br>"
                    "Y = %{y:.3f} m<br>"
                    "Z = %{z:.3f} m"
                    "<extra></extra>"
                ),

                name="Nodes"
            )
        )


    # --------------------------------------------------------
    # 6. NODE LABELS
    # --------------------------------------------------------

    if show_node_labels:

        X = []
        Y = []
        Z = []
        labels = []

        for node, coord in node_coords.items():

            X.append(coord[0])
            Y.append(coord[1])
            Z.append(coord[2])

            labels.append(str(node))


        fig.add_trace(
            go.Scatter3d(

                x=X,
                y=Y,
                z=Z,

                mode="text",

                text=labels,

                textposition="top center",

                textfont=dict(
                    size=8
                ),

                showlegend=False
            )
        )


    # --------------------------------------------------------
    # 7. SUPPORTS
    # --------------------------------------------------------

    if show_supports:

        support_nodes = []

        for node in node_tags:

            try:

                # OpenSees nodeCoord
                crd = ops.nodeCoord(node)

                if len(crd) < 3:
                    continue

                # ------------------------------------------------
                # NOTE:
                # OpenSeesPy doesn't provide a simple universal
                # "get fixity" command.
                #
                # Therefore, if you have a list of base nodes,
                # put them here.
                # ------------------------------------------------

            except:
                continue


    # --------------------------------------------------------
    # 8. AXES / LAYOUT
    # --------------------------------------------------------

    fig.update_layout(

        title=dict(
            text="OpenSeesPy 3D Structural Model",
            x=0.5
        ),

        scene=dict(

            xaxis=dict(
                title="X (m)",
                showbackground=True,
                backgroundcolor="rgb(245,245,245)",
                gridcolor="white"
            ),

            yaxis=dict(
                title="Y (m)",
                showbackground=True,
                backgroundcolor="rgb(245,245,245)",
                gridcolor="white"
            ),

            zaxis=dict(
                title="Z (m)",
                showbackground=True,
                backgroundcolor="rgb(245,245,245)",
                gridcolor="white"
            ),

            aspectmode="data",

            camera=dict(
                eye=dict(
                    x=1.6,
                    y=1.6,
                    z=1.2
                )
            )
        ),

        margin=dict(
            l=0,
            r=0,
            b=0,
            t=50
        ),

        hovermode="closest"
    )


    # --------------------------------------------------------
    # 9. SHOW
    # --------------------------------------------------------


    return fig
plot_opensees_3d()

Number of nodes: 1688
Number of elements: 2157


In [13]:
# Mass Definition
# ============================================================
# MASS ASSIGNMENT BASED ON STORY MASS - Diaphragm Masters
# ============================================================


# ------------------------------------------------------------
# Read data
# ------------------------------------------------------------


Story_Mass_Data = pd.read_excel("Load.xlsx",sheet_name="Story Mass")


# ============================================================
# STORY DEFINITIONS
# ============================================================
# Story Name  ->  Z coordinate

Story_Z = {
    "Story1": 7.1,
    "Story2": 11.3,
    "Story3": 15.5,
    "Story4": 19.4,
    "Story5": 23.3,
    "Story6": 27.2,
    "Kharposhteh": 31.8,
}


# ============================================================
# MASS ASSIGNMENT - Masters only
# ============================================================

for story_name in ["Story1","Story2","Story3","Story4","Story5","Story6"]:
    Mass_Row = Story_Mass_Data[Story_Mass_Data["Story"].astype(str).str.strip()== story_name].iloc[0]
    Total_Mass_X = float(Mass_Row["UX"]) * 1000.0
    Total_Mass_Y = float(Mass_Row["UY"]) * 1000.0
    tag = Master_Tags[story_name]
    ops.mass(tag, Total_Mass_X, Total_Mass_Y, 0.0, 0.0, 0.0, 0.0)
    print("=" * 60)
    print(f"Story       : {story_name}")
    print(f"Master      : {tag}")
    print(f"Total UX    : {Total_Mass_X:.3f} kg")
    print(f"Total UY    : {Total_Mass_Y:.3f} kg")
    print("=" * 60)
for d in ['D7-1','D7-2']:
    row = CenterMass_Data[CenterMass_Data["Diaphragm"]==d].iloc[0]
    m = float(row["Mass X (ton)"])*1000.0
    tag = Master_Tags[d]
    ops.mass(tag, m, m, 0.0, 0.0, 0.0, 0.0)
    print("=" * 60)
    print(f"Story       : Kharposhteh {d}")
    print(f"Master      : {tag}")
    print(f"Total UX    : {m:.3f} kg")
    print("=" * 60)

Story       : Story1
Master      : 9000
Total UX    : 1761455.800 kg
Total UY    : 1761455.800 kg
Story       : Story2
Master      : 9001
Total UX    : 1702846.400 kg
Total UY    : 1702846.400 kg
Story       : Story3
Master      : 9002
Total UX    : 1689572.900 kg
Total UY    : 1689572.900 kg
Story       : Story4
Master      : 9003
Total UX    : 1839708.600 kg
Total UY    : 1839708.600 kg
Story       : Story5
Master      : 9004
Total UX    : 1799930.300 kg
Total UY    : 1799930.300 kg
Story       : Story6
Master      : 9005
Total UX    : 1752607.900 kg
Total UY    : 1752607.900 kg
Story       : Kharposhteh D7-1
Master      : 9006
Total UX    : 210146.200 kg
Story       : Kharposhteh D7-2
Master      : 9007
Total UX    : 168885.000 kg


In [14]:
# Modal Analysis
# -------------------------- Modal Analysis -------------------------- #
numModes = 12
# With diaphragm, independent DOFs = 8 masters *2 =16, so request <16
try:
    eigenvalues = ops.eigen(numModes)
except:
    numModes = 8
    eigenvalues = ops.eigen(numModes)
print(f"Eigenvalues = {eigenvalues}")
omega = np.sqrt(eigenvalues)
T = 2 * np.pi / omega

for i in range(numModes):
    print(f"Mode {i+1}: T = {T[i]:.3f} sec")

Lambda = np.diag(eigenvalues)

# -------------------------- End Modal Analysis -------------------------- #
# ============================================================
# MODAL ANALYSIS - Participation
# ============================================================

totalM = float(Story_Mass_Data["UX"].sum()*1000)
master_masses = {}
for s in ['Story1','Story2','Story3','Story4','Story5','Story6']:
    master_masses[Master_Tags[s]] = float(Story_Mass_Data[Story_Mass_Data["Story"]==s]["UX"].iloc[0])*1000
for d in ['D7-1','D7-2']:
    master_masses[Master_Tags[d]] = float(CenterMass_Data[CenterMass_Data["Diaphragm"]==d]["Mass X (ton)"].iloc[0])*1000
print("="*60)
print("Modal Participation (diaphragm masters):")
print("="*60)
cumX = 0; cumY = 0
for mode in range(1, numModes+1):
    phiMrX = 0; phiMphiX = 0; phiMrY = 0; phiMphiY = 0
    for nid, mass in master_masses.items():
        try: phix = ops.nodeEigenvector(int(nid), mode, 1)
        except: phix = 0
        try: phiy = ops.nodeEigenvector(int(nid), mode, 2)
        except: phiy = 0
        phiMrX += mass*phix; phiMphiX += mass*phix*phix
        phiMrY += mass*phiy; phiMphiY += mass*phiy*phiy
    effX = (phiMrX**2)/phiMphiX if phiMphiX>1e-12 else 0
    effY = (phiMrY**2)/phiMphiY if phiMphiY>1e-12 else 0
    cumX += effX; cumY += effY
    print(f"Mode {mode:2d} T={T[mode-1]:.3f}s effX {effX/totalM*100:5.1f}% effY {effY/totalM*100:5.1f}% cumX {cumX/totalM*100:5.1f}% cumY {cumY/totalM*100:5.1f}%")

# ============================================================
# MODAL ANALYSIS
# ============================================================

numModes = 12
try:
    eigenvalues = ops.eigen(
        '-fullGenLapack',
        numModes
    )
    omega = np.sqrt(eigenvalues)
    T = 2 * np.pi / omega
    for i in range(numModes):
        print(
            f"Mode {i+1}: "
            f"lambda = {eigenvalues[i]:.6e}, "
            f"omega = {omega[i]:.4f} rad/s, "
            f"T = {T[i]:.4f} sec"
        )
except Exception as e:
    print("Eigenvalue analysis FAILED:")
    print(e)

Eigenvalues = [260.8275863098119, 938.0166074182321, 1155.7126921858587, 1321.0457227543159, 1397.154830113462, 1665.3724574683579, 1748.1815669440928, 2033.1111540957402, 2906.8215673982413, 3034.861483862697, 3531.130347671861, 5723.016939587064]
Mode 1: T = 0.389 sec
Mode 2: T = 0.205 sec
Mode 3: T = 0.185 sec
Mode 4: T = 0.173 sec
Mode 5: T = 0.168 sec
Mode 6: T = 0.154 sec
Mode 7: T = 0.150 sec
Mode 8: T = 0.139 sec
Mode 9: T = 0.117 sec
Mode 10: T = 0.114 sec
Mode 11: T = 0.106 sec
Mode 12: T = 0.083 sec
Modal Participation (diaphragm masters):
Mode  1 T=0.389s effX   0.0% effY   1.9% cumX   0.0% cumY   1.9%
Mode  2 T=0.205s effX  59.5% effY   0.0% cumX  59.5% cumY   1.9%
Mode  3 T=0.185s effX   0.0% effY   0.0% cumX  59.5% cumY   1.9%
Mode  4 T=0.173s effX  21.9% effY   0.0% cumX  81.4% cumY   1.9%
Mode  5 T=0.168s effX   4.8% effY   0.0% cumX  86.2% cumY   1.9%
Mode  6 T=0.154s effX   1.7% effY   0.0% cumX  87.9% cumY   1.9%
Mode  7 T=0.150s effX   3.2% effY   0.0% cumX  91.1% 

WARNING - the 'fullGenLapack' eigen solver is VERY SLOW. Consider using the default eigen solver.

In [15]:
# Time History Analysis
import os
import glob
import matplotlib.pyplot as plt
RECORD_FOLDER = "Scaled_Text_Records"
RESULT_FOLDER = "THA_Results_Nonlinear"
PLOTS_FOLDER = os.path.join(RESULT_FOLDER, "Plots")

os.makedirs(RESULT_FOLDER, exist_ok=True)
os.makedirs(PLOTS_FOLDER, exist_ok=True)

In [16]:
print("\nSaving model state after gravity analysis...")

ops.database("File", "GravityState.db")
ops.save(1)

print("Gravity state saved.")


Saving model state after gravity analysis...
Gravity state saved.


In [17]:
# Direction of earthquake excitation
# 1 = X
# 2 = Y
# 3 = Z
EQ_DIRECTION = 1

# Gravity acceleration
g = 9.81

# Convergence settings
MAX_ITER = 100
TOL = 1e-8
record_files = sorted(
    glob.glob(os.path.join(RECORD_FOLDER, "*.txt"))
)

print("=" * 70)
print(f"Number of earthquake records found: {len(record_files)}")
print("=" * 70)

if len(record_files) == 0:
    raise FileNotFoundError(
        f"No .txt files found in folder: {RECORD_FOLDER}"
    )

Number of earthquake records found: 14


In [18]:
def read_ground_motion(filename):

    data = np.loadtxt(filename)

    time = data[:, 0]
    acc_g = data[:, 1]

    # Convert g -> m/s²
    acc = acc_g * g

    # Calculate dt
    dt_array = np.diff(time)

    dt = np.median(dt_array)

    # Check whether dt is approximately constant
    if not np.allclose(
        dt_array,
        dt,
        rtol=1e-5,
        atol=1e-8
    ):
        print(
            f"WARNING: Non-uniform time step detected in {filename}"
        )

    return time, acc_g, acc, dt

In [19]:
def run_time_history(record_file, record_index):

    record_name = os.path.splitext(
        os.path.basename(record_file)
    )[0]

    print("\n")
    print("=" * 70)
    print(f"Running record: {record_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # Restore model to post-gravity state
    # --------------------------------------------------------

    # ops.restore returns None on success in this OpenSeesPy version - do not treat None as failure
    try:
        ops.restore(1)
    except Exception as e:
        print(f"WARNING: ops.restore failed: {e}")

    # --------------------------------------------------------
    # Read record
    # --------------------------------------------------------

    time, acc_g, acc, dt = read_ground_motion(record_file)

    n_steps = len(acc)

    total_time = time[-1]

    print(f"dt           = {dt:.6f} sec")
    print(f"Steps        = {n_steps}")
    print(f"Duration     = {total_time:.3f} sec")
    print(f"PGA          = {np.max(np.abs(acc_g)):.4f} g")
    print(f"PGA           = {np.max(np.abs(acc)):.4f} m/s²")

    # --------------------------------------------------------
    # Create unique tags
    # --------------------------------------------------------

    timeSeriesTag = 10000 + record_index
    patternTag = 20000 + record_index

    print(f"TimeSeries tag = {timeSeriesTag}")
    print(f"Pattern tag    = {patternTag}")

    # --------------------------------------------------------
    # Create Path TimeSeries
    # --------------------------------------------------------

    ops.timeSeries(
        "Path",
        timeSeriesTag,
        "-dt", dt,
        "-values", *acc,
        "-factor", 1.0
    )

    # --------------------------------------------------------
    # Apply ground motion
    # --------------------------------------------------------

    ops.pattern(
        "UniformExcitation",
        patternTag,
        EQ_DIRECTION,
        "-accel",
        timeSeriesTag
    )

    # --------------------------------------------------------
    # Analysis configuration
    # --------------------------------------------------------

    ops.wipeAnalysis()

    # Constraints
    ops.constraints("Transformation")

    # DOF numbering
    ops.numberer("RCM")

    # System solver
    ops.system("UmfPack")

    # Convergence test
    ops.test(
        "NormDispIncr",
        TOL,
        MAX_ITER,
        0,
        2
    )

    # Nonlinear solution algorithm
    ops.algorithm("NewtonLineSearch")

    # Newmark integration
    gamma = 0.5
    beta = 0.25

    ops.integrator(
        "Newmark",
        gamma,
        beta
    )

    # Transient analysis
    ops.analysis("Transient")

    # --------------------------------------------------------
    # Storage
    # --------------------------------------------------------

    results = {
        "time": [],
        "max_disp": [],
        "max_vel": [],
        "max_acc": []
    }

    # --------------------------------------------------------
    # Run analysis
    # --------------------------------------------------------

    success = True

    diaphragms_disp_1 = []
    diaphragms_disp_2 = []
    diaphragms_disp_3 = []
    diaphragms_disp_4 = []
    diaphragms_disp_5 = []
    diaphragms_disp_6 = []
    
    for step in range(n_steps):

        ok = ops.analyze(1, dt)

        if ok != 0:

            print(
                f"\nWARNING: Analysis failed at step "
                f"{step + 1}/{n_steps}"
            )

            print(
                f"Time = {ops.getTime():.6f} sec"
            )

            success = False
            break

        # ----------------------------------------------------
        # Collect response
        # --------------------------------------------------------

        current_time = ops.getTime()

        node_disp = []
        node_vel = []
        node_acc = []

        # Get all node tags
        node_tags = ops.getNodeTags()

        for node in node_tags:

            try:

                disp = ops.nodeDisp(node, EQ_DIRECTION)
                vel = ops.nodeVel(node, EQ_DIRECTION)
                acc_node = ops.nodeAccel(node, EQ_DIRECTION)

                node_disp.append(abs(disp))
                node_vel.append(abs(vel))
                node_acc.append(abs(acc_node))
            
            except:
                pass

        try:
            # Diaphragm displacements for drifts:
            disp_1 = ops.nodeDisp(9000, EQ_DIRECTION)
            disp_2 = ops.nodeDisp(9001, EQ_DIRECTION)
            disp_3 = ops.nodeDisp(9002, EQ_DIRECTION)
            disp_4 = ops.nodeDisp(9003, EQ_DIRECTION)
            disp_5 = ops.nodeDisp(9004, EQ_DIRECTION)
            disp_6 = ops.nodeDisp(9005, EQ_DIRECTION)
            diaphragms_disp_1.append(disp_1)
            diaphragms_disp_2.append(disp_2)
            diaphragms_disp_3.append(disp_3)
            diaphragms_disp_4.append(disp_4)
            diaphragms_disp_5.append(disp_5)
            diaphragms_disp_6.append(disp_6)

        except Exception as e:
            diaphragms_disp_1.append(np.nan)
            diaphragms_disp_2.append(np.nan)
            diaphragms_disp_3.append(np.nan)
            diaphragms_disp_4.append(np.nan)
            diaphragms_disp_5.append(np.nan)
            diaphragms_disp_6.append(np.nan)

        # Maximum response among all nodes
        if len(node_disp) > 0:

            results["time"].append(current_time)
            results["max_disp"].append(max(node_disp))
            results["max_vel"].append(max(node_vel))
            results["max_acc"].append(max(node_acc))

    # --------------------------------------------------------
    # Convert results to DataFrame
    # --------------------------------------------------------

    results_df = pd.DataFrame(results)

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    result_file = os.path.join(
        RESULT_FOLDER,
        f"{record_name}_response.csv"
    )

    results_df.to_csv(
        result_file,
        index=False
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    if success and len(results_df) > 0:

        max_disp = results_df["max_disp"].max()
        max_vel = results_df["max_vel"].max()
        max_acc = results_df["max_acc"].max()

        print("\nAnalysis SUCCESS")

        print(f"Maximum displacement = {max_disp:.6e} m")
        print(f"Maximum velocity     = {max_vel:.6e} m/s")
        print(f"Maximum acceleration = {max_acc:.6e} m/s²")

        print(f"Results saved to:")
        print(result_file)

    else:

        max_disp = np.nan
        max_vel = np.nan
        max_acc = np.nan

        print("\nAnalysis FAILED")


    # --------------------------------------------------------
    # Drift results
    # --------------------------------------------------------
    

    if isinstance(Story_Z, dict):
        story_heights = [
            abs(Story_Z["Story2"] - Story_Z["Story1"]),
            abs(Story_Z["Story3"] - Story_Z["Story2"]),
            abs(Story_Z["Story4"] - Story_Z["Story3"]),
            abs(Story_Z["Story5"] - Story_Z["Story4"]),
            abs(Story_Z["Story6"] - Story_Z["Story5"]),
        ]

    def safe_drift(a,b):
        return [abs(y-x) for x,y in zip(a,b) if not (np.isnan(x) or np.isnan(y))]

    Drift_1 = safe_drift(diaphragms_disp_1, diaphragms_disp_2)
    Drift_2 = safe_drift(diaphragms_disp_2, diaphragms_disp_3)
    Drift_3 = safe_drift(diaphragms_disp_3, diaphragms_disp_4)
    Drift_4 = safe_drift(diaphragms_disp_4, diaphragms_disp_5)
    Drift_5 = safe_drift(diaphragms_disp_5, diaphragms_disp_6)

    max_drift_1 = max(Drift_1)/story_heights[0] if Drift_1 and story_heights[0] else np.nan
    max_drift_2 = max(Drift_2)/story_heights[1] if Drift_2 and story_heights[1] else np.nan
    max_drift_3 = max(Drift_3)/story_heights[2] if Drift_3 and story_heights[2] else np.nan
    max_drift_4 = max(Drift_4)/story_heights[3] if Drift_4 and story_heights[3] else np.nan
    max_drift_5 = max(Drift_5)/story_heights[4] if Drift_5 and story_heights[4] else np.nan

    print(f"1st story drift is {max_drift_1 * 100} %")
    print(f"2nd story drift is {max_drift_2 * 100} %")
    print(f"3ed story drift is {max_drift_3 * 100} %")
    print(f"4th story drift is {max_drift_4 * 100} %")
    print(f"5th story drift is {max_drift_5 * 100} %")


    # --------------------------------------------------------
    # Plots for this record
    # --------------------------------------------------------

    if success and len(results_df) > 0:
        try:
            # 1. Ground motion
            plt.figure(figsize=(10, 4))
            plt.plot(time, acc, linewidth=0.8)
            plt.xlabel("Time (s)")
            plt.ylabel("Ground Acceleration (m/s2)")
            plt.title(f"Ground Motion - {record_name}")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(PLOTS_FOLDER, f"{record_name}_ground_motion.png"), dpi=150)
            plt.close()
            # 2. Roof displacement (master Story6 / D7-1)
            roof_master = Master_Tags.get('Story6', Master_Tags.get('D7-1'))
            # Roof displacement already in max_disp, but also plot master displacement if available
            plt.figure(figsize=(10, 4))
            plt.plot(results["time"], results["max_disp"], linewidth=0.8, label="Max Displacement")
            plt.xlabel("Time (s)")
            plt.ylabel("Roof Displacement (m)")
            plt.title(f"Roof Displacement - {record_name}")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(PLOTS_FOLDER, f"{record_name}_roof_disp.png"), dpi=150)
            plt.close()
            # 3. Roof velocity
            plt.figure(figsize=(10, 4))
            plt.plot(results["time"], results["max_vel"], linewidth=0.8, color="green")
            plt.xlabel("Time (s)")
            plt.ylabel("Roof Velocity (m/s)")
            plt.title(f"Roof Velocity - {record_name}")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(PLOTS_FOLDER, f"{record_name}_roof_vel.png"), dpi=150)
            plt.close()
            # 4. Roof acceleration
            plt.figure(figsize=(10, 4))
            plt.plot(results["time"], results["max_acc"], linewidth=0.8, color="red")
            plt.xlabel("Time (s)")
            plt.ylabel("Roof Acceleration (m/s2)")
            plt.title(f"Roof Acceleration - {record_name} (relative)")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(PLOTS_FOLDER, f"{record_name}_roof_acc.png"), dpi=150)
            plt.close()
        except Exception as e:
            print(f"WARNING: Plot failed for {record_name}: {e}")

    # --------------------------------------------------------
    # Remove earthquake pattern and timeSeries
    # --------------------------------------------------------

    try:
        ops.remove("loadPattern", patternTag)
    except:
        pass
    try:
        ops.remove("timeSeries", timeSeriesTag)
    except:
        pass
    # Clear analysis to avoid recorder interference
    try:
        ops.wipeAnalysis()
    except:
        pass

    return {
        "Record": record_name,
        "dt": dt,
        "Duration": total_time,
        "Steps": n_steps,
        "PGA_g": np.max(np.abs(acc_g)),
        "PGA_m_s2": np.max(np.abs(acc)),
        "Max_Displacement_m": max_disp,
        "Max_Velocity_m_s": max_vel,
        "Max_Acceleration_m_s2": max_acc,
        "Success": success,
        "Failure_Reason": "" if success else "Analysis failed",
        "TimeSeriesTag": timeSeriesTag,
        "PatternTag": patternTag
    }

In [20]:
# ============================================================
# 6. RUN ALL EARTHQUAKE RECORDS
# ============================================================

summary = []

for idx, record_file in enumerate(record_files):

    try:

        result = run_time_history(record_file, idx)

        summary.append(result)

    except Exception as e:

        print("\nERROR:")
        print(record_file)
        print(e)
        import traceback
        traceback.print_exc()

        failure_reason = str(e)
        if "tag: 10000" in failure_reason or "could not add timeseries" in failure_reason.lower():
            failure_reason = "TimeSeries tag collision - " + failure_reason
        elif "tag:" in failure_reason.lower():
            failure_reason = "Tag collision - " + failure_reason

        summary.append({
            "Record": os.path.splitext(
                os.path.basename(record_file)
            )[0],
            "dt": np.nan,
            "Duration": np.nan,
            "Steps": np.nan,
            "PGA_g": np.nan,
            "PGA_m_s2": np.nan,
            "Max_Displacement_m": np.nan,
            "Max_Velocity_m_s": np.nan,
            "Max_Acceleration_m_s2": np.nan,
            "Success": False,
            "Failure_Reason": failure_reason,
            "TimeSeriesTag": 10000 + idx,
            "PatternTag": 20000 + idx
        })




Running record: A-TMZ000_Scaled
dt           = 0.005000 sec
Steps        = 7269
Duration     = 36.340 sec
PGA          = 0.6504 g
PGA           = 6.3806 m/s²
TimeSeries tag = 10000
Pattern tag    = 20000

Analysis SUCCESS
Maximum displacement = 3.644307e-02 m
Maximum velocity     = 5.486874e-01 m/s
Maximum acceleration = 1.325080e+01 m/s²
Results saved to:
THA_Results_Nonlinear\A-TMZ000_Scaled_response.csv
1st story drift is 0.2034263342716402 %
2nd story drift is 0.2959847526852841 %
3ed story drift is 0.3639277768862626 %
4th story drift is 0.3305201439184689 %
5th story drift is 0.7140128634300118 %


Running record: A-TMZ270_Scaled
dt           = 0.005000 sec
Steps        = 7269
Duration     = 36.340 sec
PGA          = 0.4249 g
PGA           = 4.1682 m/s²
TimeSeries tag = 10001
Pattern tag    = 20001

Analysis SUCCESS
Maximum displacement = 2.188842e-02 m
Maximum velocity     = 3.072421e-01 m/s
Maximum acceleration = 1.419306e+01 m/s²
Results saved to:
THA_Results_Nonlinear\A-TMZ

In [21]:
# ============================================================
# 7. SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame(summary)

summary_file = os.path.join(
    RESULT_FOLDER,
    "THA_Summary.xlsx"
)

summary_df.to_excel(
    summary_file,
    index=False
)

print("\n")
print("=" * 70)
print("ALL EARTHQUAKE RECORDS COMPLETED")
print("=" * 70)

print(summary_df)

print("\nSummary saved to:")
print(summary_file)




ALL EARTHQUAKE RECORDS COMPLETED
             Record     dt  Duration  Steps     PGA_g   PGA_m_s2  \
0   A-TMZ000_Scaled  0.005    36.340   7269  0.650419   6.380610   
1   A-TMZ270_Scaled  0.005    36.340   7269  0.424897   4.168240   
2   ABBAR--L_Scaled  0.020    53.500   2676  1.372196  13.461243   
3   ABBAR--T_Scaled  0.020    45.980   2300  1.176586  11.542309   
4     ARC000_Scaled  0.005    29.995   6000  0.986433   9.676908   
5     ARC090_Scaled  0.005    29.995   6000  0.656793   6.443139   
6   B-ICC000_Scaled  0.005    39.995   8000  0.781389   7.665426   
7   B-ICC090_Scaled  0.005    39.995   8000  0.904460   8.872753   
8     BOL000_Scaled  0.010    55.890   5590  1.312092  12.871623   
9     BOL090_Scaled  0.010    55.890   5590  1.240292  12.167265   
10    G03000_Scaled  0.005    39.940   7989  1.145451  11.236874   
11    G03090_Scaled  0.005    39.940   7989  0.844070   8.280327   
12  H-DLT262_Scaled  0.010    99.910   9992  0.510396   5.006985   
13  H-DLT352_